# Calculus & Gradients — AI Lab Instructor Notebook

*Math for ML · Easy*

Become fluent with derivatives/gradients used in ML: chain rule, Jacobians, Hessians, and checking gradients numerically.

**Outcome.** Students can derive and implement gradients for common losses, and debug gradient bugs using finite differences.

5 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# Calculus & Gradients — FAANG-Level Lab

**Goal:** Implement and verify gradients like an ML engineer.

Key idea: *If you can’t gradient-check it, you don’t really trust it.*

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

**Why this works.** # Calculus & Gradients — Instructor (Solutions + Rationale)

## Task 2: Implement numerical gradient for scalar f(w)
*Section: Finite Differences (Gradient Checking)*

## Section 1 — Finite Differences (Gradient Checking)

### Task 1.1: Implement numerical gradient for scalar f(w)

- Use central difference: (f(w+eps e_i) - f(w-eps e_i)) / (2 eps)

**Explain:** Why is central difference more accurate than forward difference?

**Hints**
- Use central difference: (f(w+eps e_i) - f(w-eps e_i)) / (2 eps)

**Explain:** Why is central difference more accurate than forward difference?

In [ ]:
def numerical_grad(f, w, eps=1e-5):
    w = np.asarray(w, dtype=float)
    g = np.zeros_like(w)
    for i in range(w.size):
        e = np.zeros_like(w)
        e[i] = 1.0
        g[i] = (f(w + eps*e) - f(w - eps*e)) / (2*eps)
    return g

w = rng.standard_normal(5)
f = lambda v: float(np.sum(v*v))
g_num = numerical_grad(f, w)
check('grad_close', np.allclose(g_num, 2*w, atol=1e-6))

# Rationale: central difference cancels first-order error terms => more accurate than forward diff.

In [ ]:
# Checks
check('grad_close', np.allclose(g_num, 2*w, atol=1e-6))

**Why this works.** central difference cancels first-order error terms => more accurate than forward diff.

## Task 3: Gradient of MSE for linear model
*Section: Chain Rule in Code*

## Section 2 — Chain Rule in Code

### Task 2.1: Gradient of MSE for linear model

Model: y_hat = Xw
Loss: L(w) = (1/n) * sum_i (y_hat_i - y_i)^2

- Let r = Xw - y
- grad = (2/n) * X^T r

**FAANG gotcha:** shape mismatches; keep w as (d,) and X as (n,d).

**Hints**
- Let r = Xw - y
grad = (2/n) * X^T r

In [ ]:
def mse_loss_and_grad(X, y, w):
    r = X @ w - y
    n = X.shape[0]
    loss = float(np.mean(r*r))
    grad = (2.0/n) * (X.T @ r)
    return loss, grad

n,d = 20,4
X = rng.standard_normal((n,d))
w = rng.standard_normal(d)
y = rng.standard_normal(n)
loss, grad = mse_loss_and_grad(X,y,w)
f = lambda v: mse_loss_and_grad(X,y,v)[0]
g_num = numerical_grad(f, w)
check('mse_grad', np.allclose(grad, g_num, atol=1e-5))

# Rationale: chain rule + linearity => grad is X^T residual. Shapes: X^T (d,n) @ r (n,) => (d,)

In [ ]:
# Checks
check('mse_grad', np.allclose(grad, g_num, atol=1e-5))

**Why this works.** chain rule + linearity => grad is X^T residual. Shapes: X^T (d,n) @ r (n,) => (d,)

## Task 4: Binary cross-entropy gradient
*Section: Logistic Regression (Core Interview Gradient)*

## Section 3 — Logistic Regression (Core Interview Gradient)

### Task 3.1: Binary cross-entropy gradient

Given labels y in {0,1}.
p = sigmoid(Xw)
Loss = -(1/n) * sum(y log p + (1-y) log(1-p))

- sigmoid(z)=1/(1+exp(-z))
- grad = (1/n) * X^T (p - y)
- add numerical stability for logs (clip p)

**Hints**
- sigmoid(z)=1/(1+exp(-z))
grad = (1/n) * X^T (p - y)
add numerical stability for logs (clip p)

In [ ]:
def sigmoid(z):
    # stable-ish sigmoid
    return 1.0 / (1.0 + np.exp(-z))

def logreg_loss_and_grad(X, y, w):
    z = X @ w
    p = sigmoid(z)
    eps = 1e-12
    p_clip = np.clip(p, eps, 1.0 - eps)
    loss = float(-np.mean(y*np.log(p_clip) + (1-y)*np.log(1-p_clip)))
    grad = (X.T @ (p - y)) / X.shape[0]
    return loss, grad

n,d = 50,3
X = rng.standard_normal((n,d))
w = rng.standard_normal(d)
y = rng.integers(0,2,size=n)
loss, grad = logreg_loss_and_grad(X,y,w)
f = lambda v: logreg_loss_and_grad(X,y,v)[0]
g_num = numerical_grad(f, w)
check('logreg_grad', np.allclose(grad, g_num, atol=1e-5))

# Rationale: derivative of BCE w.r.t logits gives (p-y); then chain through z=Xw => X^T(p-y).

In [ ]:
# Checks
check('logreg_grad', np.allclose(grad, g_num, atol=1e-5))

**Why this works.** derivative of BCE w.r.t logits gives (p-y); then chain through z=Xw => X^T(p-y).

## Task 5: Compute Hessian of f(w)=sum(w^2) numerically
*Section: Jacobian/Hessian Intuition*

## Section 4 — Jacobian/Hessian Intuition

### Task 4.1: Compute Hessian of f(w)=sum(w^2) numerically

![Hessian matrix formula with second partial derivatives](/images/w2-d2-hessian-matrix-formula.png)

- Hessian of sum(w^2) is 2I
- Use numerical_grad on each component of grad

This is mainly about *shape thinking*: Hessian is (d,d).

**Hints**
- Hessian of sum(w^2) is 2I
Use numerical_grad on each component of grad

In [ ]:
def numerical_hessian(f, w, eps=1e-4):
    w = np.asarray(w, dtype=float)
    d = w.size
    H = np.zeros((d,d))
    # grad_i = d f / d w_i
    for i in range(d):
        def fi(v):
            # partial derivative approx for component i
            e = np.zeros(d); e[i]=1.0
            return (f(v + eps*e) - f(v - eps*e)) / (2*eps)
        H[i,:] = numerical_grad(fi, w, eps=eps)
    return H

w = rng.standard_normal(4)
f = lambda v: float(np.sum(v*v))
H = numerical_hessian(f, w)
check('H_shape', H.shape == (4,4))
check('H_close', np.allclose(H, 2*np.eye(4), atol=1e-3))

# Rationale: Hessian captures curvature; in optimization it affects step sizes and convergence.

In [ ]:
# Checks
check('H_shape', H.shape == (4,4))
check('H_close', np.allclose(H, 2*np.eye(4), atol=1e-3))

**Why this works.** Hessian captures curvature; in optimization it affects step sizes and convergence.

![Hessian matrix formula with second partial derivatives](/images/w2-d2-hessian-matrix-formula.png)